# Exploratory Analysis: Stimulus Generalization Experiment

This notebook provides an end-to-end walkthrough of the analysis pipeline for the
go/no-go stimulus generalization experiment. It:

1. Generates simulated trial-level data
2. Demonstrates data cleaning and exclusion
3. Computes and visualizes generalization gradients
4. Builds the modeling dataset with history-based features
5. Fits starter logistic-regression models

All functions are imported from the `analysis` package so that the same code
runs identically in the pipeline CLI (`run_pipeline.py`) and here in the notebook.

In [ ]:
import sys, os
from pathlib import Path

# Ensure the repo root is on the Python path so we can import `analysis.*`
REPO_ROOT = Path(os.getcwd()).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

print(f"Repo root: {REPO_ROOT}")

## 1. Generate simulated data

We create synthetic trial-level data that mirrors the structure of the real
experiment export. The simulated data includes:

- 10 participants, each completing ~220 trials across phases
- Probe stimuli at 11 positions on the normalized [0, 1] axis
- A Gaussian generalization gradient centred on the target (S+ = 0.50)
- Realistic RT distributions and reinforcement contingencies

In [ ]:
def generate_simulated_data(
    n_participants: int = 10,
    seed: int = 42,
) -> pd.DataFrame:
    """Create a realistic simulated dataset for pipeline testing."""
    rng = np.random.default_rng(seed)

    probe_positions = [0.10, 0.18, 0.26, 0.34, 0.42, 0.50, 0.58, 0.66, 0.74, 0.82, 0.90]
    target_x = 0.50
    sigma = 0.15  # width of the generalization gradient

    phases = [
        ("practice", 10),
        ("baseline_training", 60),
        ("early_probes", 55),       # mix of training + probes
        ("discrimination", 40),
        ("late_probes", 55),
    ]

    rows = []
    for p_idx in range(n_participants):
        pid = f"P{p_idx+1:03d}"
        trial_global = 0
        # Per-participant gradient noise
        participant_sigma = sigma + rng.normal(0, 0.02)

        for phase_id, n_trials in phases:
            for t_in_phase in range(n_trials):
                # Decide trial type and stimulus
                if phase_id == "practice":
                    stim_x = target_x
                    trial_type = "training"
                elif phase_id in ("baseline_training", "discrimination"):
                    stim_x = target_x
                    trial_type = "training"
                else:
                    # Probe phases: ~60% training, ~40% probe
                    if rng.random() < 0.6:
                        stim_x = target_x
                        trial_type = "training"
                    else:
                        stim_x = rng.choice(probe_positions)
                        trial_type = "probe"

                distance = stim_x - target_x
                abs_distance = abs(distance)
                sim_gauss = np.exp(-0.5 * (distance / participant_sigma) ** 2)
                sim_exp = np.exp(-abs_distance / participant_sigma)

                # Response probability based on Gaussian similarity
                # with learning effect (increases over trials)
                learning_factor = min(1.0, trial_global / 100)
                p_respond = sim_gauss * (0.3 + 0.7 * learning_factor)
                p_respond = np.clip(p_respond + rng.normal(0, 0.02), 0.01, 0.99)
                responded = rng.random() < p_respond

                is_target = trial_type == "training"
                reinf_available = is_target
                reinf_delivered = is_target and responded

                rt = np.nan
                if responded:
                    # RT ~ 300 + 200 * distance + noise
                    rt = 300 + 200 * abs_distance + rng.exponential(80)
                    rt = max(50, rt)  # allow a few implausibly fast for testing

                label = "target" if stim_x == target_x else f"probe_{stim_x:.2f}"

                rows.append({
                    "participant_id": pid,
                    "external_id": f"ext_{pid}",
                    "session_id": f"sess_{pid}_001",
                    "study_config_id": "config_v1",
                    "condition_id": "cond_01",
                    "condition_name": "standard",
                    "phase_id": phase_id,
                    "block_index": trial_global // 20,
                    "trial_index_global": trial_global,
                    "trial_index_within_phase": t_in_phase,
                    "stimulus_x_normalized": stim_x,
                    "stimulus_render_value": stim_x * 180,  # degrees
                    "stimulus_label": label,
                    "signed_distance_from_target": distance,
                    "absolute_distance_from_target": abs_distance,
                    "similarity_gaussian": sim_gauss,
                    "similarity_exponential": sim_exp,
                    "trial_type": trial_type,
                    "reinforcement_available": reinf_available,
                    "reinforcement_delivered": reinf_delivered,
                    "feedback_type": "correct" if reinf_delivered else ("none" if not is_target else "omission"),
                    "response_occurred": responded,
                    "response_count": int(responded),
                    "first_response_rt_ms": rt,
                    "all_response_timestamps_ms": f"[{rt:.1f}]" if responded else "[]",
                    "trial_start_ts": f"2025-06-01T10:00:{trial_global:05.1f}Z",
                    "stimulus_onset_ts": f"2025-06-01T10:00:{trial_global + 0.5:05.1f}Z",
                    "feedback_onset_ts": f"2025-06-01T10:00:{trial_global + 2.0:05.1f}Z",
                    "trial_end_ts": f"2025-06-01T10:00:{trial_global + 3.0:05.1f}Z",
                    "browser_tz_offset": -300,
                    "is_resumed": False,
                    "is_attention_check": False,
                    "exclusion_flag": False,
                    "config_hash": "abc123",
                    "random_seed": seed,
                })
                trial_global += 1

    return pd.DataFrame(rows)

df_raw = generate_simulated_data()
print(f"Simulated dataset: {len(df_raw)} trials, {df_raw['participant_id'].nunique()} participants")
print(f"Phases: {df_raw['phase_id'].unique().tolist()}")
df_raw.head(3)

## 2. Data cleaning and exclusion

Apply the full exclusion pipeline from `analysis.clean`. This removes pilots,
flagged participants, practice trials, implausibly fast RTs, and participants
with excessive misses during training.

In [ ]:
from analysis.clean import apply_all_exclusions

df_clean, excl_summary = apply_all_exclusions(df_raw)

print("Exclusion summary:")
for k, v in excl_summary.items():
    print(f"  {k}: {v}")

print(f"\nClean dataset: {len(df_clean)} trials, {df_clean['participant_id'].nunique()} participants")

## 3. Descriptive summaries

Compute trial-level, participant-level, and phase-level summaries.

In [ ]:
from analysis.summaries import trial_level_summary, participant_summary, phase_summary

df_clean = trial_level_summary(df_clean)

p_summ = participant_summary(df_clean)
print("Participant summary:")
display(p_summ[["n_trials", "n_training", "n_probe",
                "response_rate_training", "response_rate_probe",
                "mean_rt"]].describe().round(2))

ph_summ = phase_summary(df_clean)
print("\nPhase summary (first 10 rows):")
display(ph_summ.head(10))

## 4. Generalization gradients

The generalization gradient is the key dependent measure: P(response) as a
function of stimulus position along the normalized dimension. We expect a
peaked function centred on (or near) the S+ with decreasing probability at
more distant probe positions.

In [ ]:
from analysis.gradients import (
    compute_gradient,
    compute_gradient_by_block,
    group_mean_gradient,
)

# --- Group-level gradient ---
group_grad = group_mean_gradient(df_clean)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(group_grad["stimulus_x"], group_grad["response_probability"],
        "o-", color="#d7191c", markersize=6, linewidth=1.8, label="Group mean")
ax.fill_between(group_grad["stimulus_x"],
                group_grad["ci_lower"], group_grad["ci_upper"],
                alpha=0.2, color="#d7191c", label="95% CI")
ax.axvline(0.50, ls="--", color="gray", alpha=0.5, label="S+ (target)")
ax.set_xlabel("Stimulus position (normalized)")
ax.set_ylabel("P(response)")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Group generalization gradient")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(group_grad.to_string(index=False))

In [ ]:
# --- Individual gradients (first 3 participants) ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, pid in zip(axes, df_clean["participant_id"].unique()[:3]):
    pdata = df_clean[df_clean["participant_id"] == pid]
    grad = compute_gradient(pdata)
    ax.plot(grad["stimulus_x"], grad["response_probability"],
            "o-", color="#2c7bb6", markersize=5)
    ax.fill_between(grad["stimulus_x"], grad["ci_lower"], grad["ci_upper"],
                    alpha=0.2, color="#2c7bb6")
    ax.axvline(0.50, ls="--", color="gray", alpha=0.5)
    ax.set_title(pid)
    ax.set_xlabel("Stimulus position")
    ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel("P(response)")
fig.suptitle("Individual generalization gradients", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Gradient evolution over blocks (first participant) ---
pid = df_clean["participant_id"].unique()[0]
pdata = df_clean[df_clean["participant_id"] == pid]
gbb = compute_gradient_by_block(pdata)

pivot = gbb.pivot_table(index="block", columns="stimulus_x",
                        values="response_probability", aggfunc="mean")

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, aspect="auto", origin="lower",
               cmap="YlOrRd", vmin=0, vmax=1, interpolation="nearest")
ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns], rotation=45, ha="right")
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels([int(b) for b in pivot.index])
ax.set_xlabel("Stimulus position")
ax.set_ylabel("Block")
ax.set_title(f"Gradient evolution -- {pid}")
fig.colorbar(im, ax=ax, label="P(response)")
plt.tight_layout()
plt.show()

## 5. Modeling dataset

Build the feature-rich modeling dataset. This adds lagged responses,
cumulative reinforcement history, rolling rates, and similarity-weighted
reinforcement features that downstream models consume.

In [ ]:
from analysis.modeling_dataset import build_modeling_dataset, ModelingConfig

config = ModelingConfig()
df_model = build_modeling_dataset(df_clean, config)

print(f"Modeling dataset shape: {df_model.shape}")
print(f"\nNew feature columns:")
new_cols = [c for c in df_model.columns if c not in df_clean.columns]
for c in sorted(new_cols):
    print(f"  {c}")

print(f"\nSample of history features (first participant, trials 95-105):")
pid = df_model["participant_id"].unique()[0]
display(
    df_model.loc[
        (df_model["participant_id"] == pid) &
        (df_model["trial_index_global"].between(95, 105)),
        ["trial_index_global", "stimulus_x_normalized", "response_occurred",
         "reinforcement_delivered", "cumulative_reinforcers_at_target",
         "cumulative_similarity_weighted_reinforcement",
         "rolling_response_rate", "lagged_response_1",
         "trials_since_last_reinforcement"]
    ].reset_index(drop=True)
)

## 6. Starter models

Fit two logistic regression models as a first-pass approximation:

1. **Simple model**: `P(response) ~ similarity_gaussian + cumulative_similarity_weighted_reinforcement + distance + block`
2. **History-weighted model**: Adds rolling rates, lagged responses, and reinforcement history

These are *not* mechanistic diffusion models -- they are descriptive baselines.
See `starter_model.py` for detailed comments on extending to a DDM.

In [ ]:
from analysis.starter_model import fit_simple_model, fit_history_weighted_model

OUTPUT_DIR = str(REPO_ROOT / "outputs")

print("--- Simple model ---\n")
simple_model = fit_simple_model(df_model, output_dir=OUTPUT_DIR)

# Show the saved plot inline
from IPython.display import Image, display as ipy_display
simple_plot = Path(OUTPUT_DIR) / "simple_model_gradient.png"
if simple_plot.exists():
    ipy_display(Image(filename=str(simple_plot), width=500))

In [ ]:
print("--- History-weighted model ---\n")
history_model = fit_history_weighted_model(df_model, output_dir=OUTPUT_DIR)

history_plot = Path(OUTPUT_DIR) / "history_model_gradient.png"
if history_plot.exists():
    ipy_display(Image(filename=str(history_plot), width=500))

## 7. RT by distance from target

Reaction times should increase with distance from the S+. This is a basic
sanity check and is also informative for parameterizing a drift-diffusion model
where drift rate decreases with distance.

In [ ]:
responded = df_clean[df_clean["response_occurred"].astype(bool)].copy()
responded["abs_dist"] = pd.to_numeric(responded["absolute_distance_from_target"], errors="coerce")
responded["rt"] = pd.to_numeric(responded["first_response_rt_ms"], errors="coerce")

fig, ax = plt.subplots(figsize=(5, 3.5))
bins = pd.cut(responded["abs_dist"], bins=8)
rt_by_dist = responded.groupby(bins, observed=True)["rt"].agg(["mean", "sem"])
midpoints = [iv.mid for iv in rt_by_dist.index]
ax.errorbar(midpoints, rt_by_dist["mean"], yerr=1.96 * rt_by_dist["sem"],
            marker="o", capsize=3, color="#2c7bb6")
ax.set_xlabel("Distance from S+ (normalized)")
ax.set_ylabel("RT (ms)")
ax.set_title("Reaction time by distance from target")
plt.tight_layout()
plt.show()

## Next steps

- Replace simulated data with real experiment export (`load_trials("path/to/trials.csv")`)
- Run the full pipeline from the command line: `python -m analysis.run_pipeline data/trials.csv`
- Extend the starter logistic regression to a drift-diffusion model (see comments in `starter_model.py`)
- Compare Gaussian vs. exponential generalization kernels
- Add hierarchical (mixed-effects) structure for participant-level parameters